<!-- cabecera-entorno -->
## Antes de empezar

**Clase 4 · EDA: estadística, GroupBy y análisis univariado** — Bloque 3 · Reto. Este cuaderno lo
recorre **usted solo**, leyendo: cada tarea trae la explicación y los comandos que necesita. El
profesor circula por el salón resolviendo dudas. Es el entregable de la clase.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `reto.ipynb` como
`reto_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El cuaderno se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/evaluaciones_agropecuarias.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 4 · Reto — El campo colombiano en tres variables

**Dataset:** `../datos/evaluaciones_agropecuarias.csv` (Evaluaciones Agropecuarias Municipales,
Ministerio de Agricultura, vía datos.gov.co)
**Consigna completa:** `README.md`

20.000 filas x 17 columnas. Una fila = **un cultivo, en un municipio, en un año** (2017 o 2018).
32 departamentos, 13 grupos de cultivo.

Las tres variables del reto:

| Columna | Qué es | Unidad |
|---------|--------|--------|
| `producci_n_t` | Producción cosechada | toneladas |
| `rea_sembrada_ha` | Área sembrada | hectáreas |
| `rendimiento_t_ha` | Rendimiento: producción dividida por área | toneladas por hectárea |

Y las que se usan para agrupar:

| Columna | Qué es |
|---------|--------|
| `departamento` | Uno de 32 departamentos |
| `municipio` | Municipio |
| `grupo_de_cultivo` | 13 grupos: FRUTALES, CEREALES, OLEAGINOSAS, ... |
| `ciclo_de_cultivo` | TRANSITORIO, PERMANENTE o ANUAL |
| `cultivo` | Nombre del cultivo |

**Tres advertencias antes de escribir la primera línea:**

1. **Los nombres de columna están mal escritos y son los reales.** `rea_sembrada_ha` perdió la "á" de
   "área" y `producci_n_t` perdió la "ó" de "producción" al exportarse desde datos.gov.co. Cópielos de
   la salida de la celda de reconocimiento; no los teclee.
2. **Todo el texto está en MAYÚSCULAS y sin tildes.** Filtrar por `'Antioquia'` devuelve cero filas.
3. **Una de las tres variables tiene valores faltantes.** El paso 1 del marco existe justamente para
   que aparezcan antes de que arruinen un cálculo.

## Cómo se recorre este cuaderno

Usted trabaja solo. Nadie va a dictar los pasos desde el tablero, así que cada tarea trae todo lo que
necesita para resolverse leyendo:

| Parte de la tarea | Qué contiene |
|-------------------|--------------|
| **La pregunta** | Lo que hay que responder, escrito en español |
| **El concepto** | Qué técnica aplica y por qué esa y no otra |
| **Los comandos** | Las instrucciones exactas que va a usar, escritas de forma genérica |
| **Lo que decide usted** | Qué columna, qué valor, qué operación. Ahí no hay respuesta escrita |
| **La celda de código** | Los pasos numerados en comentarios. Usted escribe las líneas |
| **La comprobación** | `comprobar('TN', ...)` le dice si el número es el correcto, sin mostrárselo |

**Por qué esto sigue siendo un reto y no una copia.** En el demo aplicó el marco de 5 pasos a una
variable de consumo de agua. Aquí lo aplica a tres variables de producción agrícola que no ha visto
nunca. Le damos el camino —los comandos, la técnica—, pero el camino lo recorre usted: elige la
columna, elige la operación, **clasifica la forma de la distribución** y **decide qué hacer con los
outliers**. La técnica se guía; el criterio no se guía, y el criterio es lo que se evalúa.

**Las celdas `comprobar(...)`** comparan una huella digital de su resultado con la esperada. Si
coinciden, dicen `CORRECTO`; si no, dan una pista dirigida al error más probable. Nunca muestran la
respuesta: escribir cualquier cosa hasta que pase es engañarse en el propio entregable.

**Las celdas `Tu respuesta:`** no llevan código. Son las que se leen en la dimensión Saber. Un
cuaderno con las diez tareas correctas y ninguna frase escrita está a medias.

**El recorrido:**

| Parte | Qué se hace | Tareas |
|-------|-------------|--------|
| 1 | Paso 1 del marco: identificar | T1 |
| 2 | Pasos 2 y 3 sobre `producci_n_t`, a mano | T2, T3 |
| 3 | Pasos 4 y 5 sobre `producci_n_t`: forma y outliers | T4, T5 |
| 4 | Las otras dos variables, con la función ya escrita | T6, T7 |
| 5 | GroupBy: comparar entre grupos | T8, T9 |
| 6 | Una pregunta que usted arma sola | T10 |

Las partes 5 y 6, la tabla resumen y la reflexión se terminan en casa si no alcanza el tiempo en el
salón. **Al final hay un punto de control** que dice cuántas de las diez tareas quedaron correctas.

---

## Paso 0 · Cargar los datos

**El concepto.** `pd.read_csv()` lee el archivo del disco y lo copia a la memoria como un
**DataFrame** (la tabla de pandas). Los dos puntos del principio de la ruta son la instrucción: `../`
significa "suba un nivel desde la carpeta donde está este cuaderno", y lo que sigue es la carpeta de
datos y el nombre del archivo. La ruta es relativa **al cuaderno**, no a la carpeta abierta en VSCode.

Este archivo, a diferencia del de Empocaldas, **no** tiene el problema de los separadores de miles: se
puede cargar sin `dtype=str`. Pero sí tiene faltantes, y eso lo descubre el paso 1.

**Los comandos.**

```python
df = pd.read_csv('ruta/al/archivo.csv')
df.shape       # (filas, columnas)
df.head()      # las primeras 5 filas
df.columns.tolist()   # los nombres reales, para copiar y pegar
```

Las dos celdas de abajo ya están escritas. Ejecútelas y confirme que dice 20.000 filas y 17 columnas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Este cuaderno vive en clase04/reto/
df = pd.read_csv('../datos/evaluaciones_agropecuarias.csv')

print(f'Filas: {df.shape[0]:,}   Columnas: {df.shape[1]}')
print()
print('Nombres reales de las columnas (copielos de aqui, no los teclee):')
print(df.columns.tolist())
df.head()

In [ ]:
# Verificador de las tareas. Ejecute esta celda una vez y siga adelante.
# No hace falta entenderla hoy: es andamiaje del curso, no materia de la clase.
import hashlib

_RESULTADOS = {}

_PISTAS = {
    "T1": "isna() marca los faltantes y sum() los cuenta. Se pide sobre UNA columna, no sobre el DataFrame completo: df['columna'].isna().sum(). De las tres variables del reto, solo una tiene faltantes, y no es ninguna de las dos primeras.",
    "T2": "Son dos numeros y una division: media entre mediana, en ese orden. Si le da un numero menor que 1 los invirtio. Y ojo: se pide la razon, no la diferencia.",
    "T3": "IQR = Q3 - Q1, y los cuartiles se piden con .quantile() en fraccion: 0.75 y 0.25, no 75 y 25. Es un solo numero, no una pareja.",
    "T4": "Compare en el histograma la linea roja (media) con la verde (mediana), y contrastelo con la razon que calculo en T2. El lado hacia el que se estira la cola larga es el del sesgo. Una sola palabra, en minuscula, de las cuatro de la lista.",
    "T5": "Primero los dos limites: Q1 - 1.5*IQR y Q3 + 1.5*IQR. Despues la mascara con | entre las dos condiciones, cada una entre parentesis. Se pide el CONTEO de filas marcadas, no el porcentaje ni la tabla.",
    "T6": "La funcion devuelve un diccionario. La razon esta guardada bajo la clave 'razon': se saca con resultado_area['razon']. Si le dio None, no guardo el resultado de la funcion en una variable.",
    "T7": "Igual que T6, pero sobre la tercera variable. Si el numero le sale muy parecido al de T6, revise que llamo la funcion sobre la columna correcta: rendimiento, no area.",
    "T8": "Son dos cosas en una lista: el nombre del grupo (texto, tal como esta en el dato) y su produccion media. Se separa por grupo_de_cultivo, se resume producci_n_t, con .mean(). El nombre sale con .idxmax() y el valor con .max().",
    "T9": "Se pide la MEDIANA, no la media, porque la variable esta sesgada. Se separa por ciclo_de_cultivo y se resume rendimiento_t_ha. Son solo tres ciclos: si su lista tiene mas elementos, agrupo por la columna equivocada.",
    "T10": "Son dos groupby encadenados por un filtro. El primero define QUIENES entran (suma de produccion por departamento, los 5 mas altos); el filtro se queda con esas filas del DataFrame original; el segundo calcula la mediana del rendimiento solo dentro de ellas. Si le sale un departamento que no esta entre los cinco de mayor produccion, se salto el filtro.",
}

_ESPERADO = {
    "T1": "ce0b01b554",
    "T2": "56f272a901",
    "T3": "1f565ae441",
    "T4": "d6c403091d",
    "T5": "ef12da1289",
    "T6": "6c5369f607",
    "T7": "56ee5d1d15",
    "T8": "183c87b2ff",
    "T9": "cdbc2df082",
    "T10": "f56867040f",
}


def _firma(valor):
    """Reduce un resultado a un texto reproducible, sin importar como se calculo."""
    if isinstance(valor, (list, tuple)):
        return "lista|" + "|".join(_firma(v) for v in valor)
    if isinstance(valor, pd.DataFrame):
        partes = ["DataFrame", str(valor.shape), str([str(c) for c in valor.columns]),
                  str([str(i) for i in valor.index])]
        for columna in valor.columns:
            serie = valor[columna]
            if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
                partes.append(f"{columna}:{[str(v) for v in serie.tolist()]}")
            else:
                partes.append(f"{columna}:{round(float(serie.sum()), 4)}")
        return "|".join(partes)
    if isinstance(valor, pd.Series):
        return "|".join(["Series", str(len(valor)), str([str(i) for i in valor.index]),
                         str([str(v) for v in valor.tolist()])])
    if not isinstance(valor, str):
        try:
            return f"numero|{round(float(valor), 4)}"
        except (TypeError, ValueError):
            pass
    return f"otro|{valor!r}"


def _huella(valor):
    return hashlib.sha256(_firma(valor).encode("utf-8")).hexdigest()[:10]


def comprobar(clave, valor):
    """Dice si el resultado es el correcto, sin revelar cual era."""
    _RESULTADOS[clave] = False
    if valor is None:
        print(f"[{clave}] Sin resolver todavia: la variable sigue valiendo None.")
        return
    if isinstance(valor, pd.DataFrame):
        print(f"[{clave}] Usted produjo un DataFrame de {valor.shape[0]} filas "
              f"y {valor.shape[1]} columnas.")
    elif isinstance(valor, pd.Series):
        print(f"[{clave}] Usted produjo una Series de {len(valor)} elementos, tipo {valor.dtype}.")
    elif isinstance(valor, (list, tuple)):
        print(f"[{clave}] Usted produjo: {list(valor)}")
    else:
        print(f"[{clave}] Usted produjo: {valor!r}")
    if _huella(valor) == _ESPERADO[clave]:
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO.")
    else:
        print(f"[{clave}] Todavia no coincide.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")


_ORDEN = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "T10"]
_ESTADO_OK = "correcta"
_ETIQUETA = "tareas correctas"


def resumen_puntos_de_control():
    """Estado de todo lo que se comprueba en este cuaderno."""
    print("Punto de control final")
    print("-" * 46)
    for clave in _ORDEN:
        estado = _ESTADO_OK if _RESULTADOS.get(clave) else "pendiente"
        print(f"  {clave}: {estado}")
    logrados = sum(1 for c in _ORDEN if _RESULTADOS.get(c))
    print("-" * 46)
    print(f"{logrados} de {len(_ORDEN)} {_ETIQUETA}.")


print("Verificador listo. Se comprueba con comprobar('T1', su_variable).")

### Paso 0.1 · Reconocimiento: mire los datos antes de calcular

**El concepto.** El error más frecuente de esta clase no es de sintaxis: es de supuesto. Se escribe
`'Antioquia'`, el dato dice `'ANTIOQUIA'`, el filtro devuelve cero filas y el código se ve impecable.
`.unique()` devuelve los valores que **realmente** existen en una columna, sin repeticiones, y
`.nunique()` cuenta cuántos son.

**Los comandos.**

```python
df['columna'].unique().tolist()   # los valores distintos, como estan escritos
df['columna'].nunique()           # cuantos distintos hay
df.isna().sum()                   # faltantes por columna, en todo el DataFrame
df.info()                         # tipos y no nulos por columna: es el paso 1 del marco
```

Estas dos celdas también están escritas. Ejecútelas y **no cierre la salida**: de aquí va a copiar los
nombres exactos.

In [ ]:
print('Ciclos de cultivo:', df['ciclo_de_cultivo'].unique().tolist())
print()
print('Grupos de cultivo:', df['grupo_de_cultivo'].nunique(), 'distintos')
print(sorted(df['grupo_de_cultivo'].unique().tolist()))
print()
print('Departamentos:', df['departamento'].nunique())
print('Anios:', sorted(df['a_o'].unique().tolist()))

In [ ]:
df.info()

---

## Parte 1 · Paso 1 del marco: identificar

**Qué se practica aquí.** Mirar la variable **antes** de calcular nada sobre ella.

**El concepto.** Una media calculada sobre una columna a la que le falta el 30% de los datos no
significa lo mismo que una calculada sobre la columna completa, y ninguna función se lo va a avisar:
pandas ignora los faltantes en silencio al promediar. Por eso el paso 1 va primero y no es opcional.

**Qué es `NaN`.** *Not a Number*: la marca de "aquí no hay valor". No es un cero ni una cadena vacía,
es la ausencia de dato. En este dataset la diferencia importa: una producción de 0 toneladas dice que
ese cultivo no produjo nada; un `NaN` dice que no sabemos cuánto produjo.

### Tarea 1 · Los faltantes

**La pregunta.** ¿Cuál de las tres variables del reto tiene valores faltantes, y cuántos?

**El concepto.** `.isna()` devuelve una máscara de `True`/`False`, con un `True` donde falta el valor.
Como en pandas `True` vale 1 y `False` vale 0, `.sum()` sobre esa máscara **cuenta** los faltantes.
Es el mismo truco de contar máscaras de la clase 2.

**Los comandos.**

```python
df.isna().sum()              # faltantes de TODAS las columnas, de un golpe
df['columna'].isna().sum()   # faltantes de una sola columna
```

**Lo que decide usted.** Cuál de las tres columnas del reto es la que tiene faltantes. Mírelas las
tres antes de escribir el número.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Imprima los faltantes de todas las columnas.
# 2. Guarde en faltantes_rendimiento cuantos faltantes tiene la variable de rendimiento.
# 3. Imprima que porcentaje del total representan.

faltantes_rendimiento = None

In [ ]:
comprobar('T1', faltantes_rendimiento)

**Tu respuesta:** ¿por qué esa variable y no las otras dos es la que tiene huecos? Fíjese en cómo se
calcula el rendimiento y en qué pasaría si el área sembrada de un registro fuera cero.

*Tu respuesta:*

---

## Parte 2 · Pasos 2 y 3 sobre `producci_n_t`

**Qué se practica aquí.** El centro y la dispersión de una variable, calculados a mano. La primera
variable se hace paso por paso; las otras dos, en la parte 4, con la función que ya está escrita.

**El concepto: las tres medidas de centro.**

- **Media**: la suma dividida entre el número de datos. Se la lleva cualquier valor extremo.
- **Mediana**: el valor del medio cuando se ordenan los datos. Inmune a los extremos.
- **Moda**: el que más se repite. La única que funciona con variables categóricas.

**La regla operativa: razón = media / mediana.** Si se aparta más del 20% de 1 (mayor que 1,2 o menor
que 0,8), la distribución está sesgada y la mediana es más honesta. Es la regla del salario del CEO:
nueve empleados con 3 millones y un gerente con 300 dan una media de 32,7 millones que no describe a
nadie.

**El concepto: dispersión.**

- **Desviación estándar** (`.std()`): qué tan regados están los datos, en las unidades originales. Es
  el arquero con las flechas apiñadas contra el que las tiene desparramadas.
- **Varianza** (`.var()`): la desviación estándar al cuadrado. **No se reporta**, porque está en
  unidades al cuadrado y eso no significa nada físico.
- **Percentiles y cuartiles**: Q1 es el percentil 25, Q2 es la mediana, Q3 es el percentil 75.
- **IQR = Q3 - Q1**: el rango donde vive el 50% central. Es la medida de dispersión que ignora los
  extremos.

**Los comandos de toda esta parte.**

```python
df['columna'].mean()          df['columna'].std()
df['columna'].median()        df['columna'].var()
df['columna'].mode()[0]       df['columna'].quantile(0.25)
df['columna'].describe()      df['columna'].min()   df['columna'].max()
```

### Tarea 2 · El centro de `producci_n_t`

**La pregunta.** ¿Dónde está el centro de la producción agrícola colombiana, y qué tan de acuerdo
están la media y la mediana?

**El concepto.** Calcular las dos medidas y dividirlas. El número que sale no es un trámite: es lo que
decide **cuál de las dos se reporta** en el resto del análisis.

**Los comandos.**

```python
media = df['columna'].mean()
mediana = df['columna'].median()
razon = media / mediana
```

**Lo que decide usted.** El orden de la división. Media entre mediana, no al revés: la convención
existe para que un número mayor que 1 signifique siempre lo mismo, sesgo a la derecha.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Calcule media, mediana y moda de producci_n_t, e imprimalas.
# 2. Guarde la razon media/mediana en razon_produccion e imprimala.

razon_produccion = None

In [ ]:
# El redondeo se hace aqui, para que el resultado no dependa de cuantos decimales imprima usted.
comprobar('T2', None if razon_produccion is None else round(razon_produccion, 2))

**Tu respuesta:** traduzca esa razón a una frase sobre el campo colombiano. ¿Qué significa que la
media sea tantas veces la mediana, si cada fila es un cultivo en un municipio?

*Tu respuesta:*

### Tarea 3 · La dispersión de `producci_n_t`

**La pregunta.** ¿Qué tan regada está la producción? En particular, ¿cuál es el rango donde vive el
50% central de los registros?

**El concepto.** El IQR es la distancia entre el cuartil 3 y el cuartil 1. Mide dispersión igual que
la desviación estándar, pero **sin dejarse arrastrar por los extremos**: los cuartiles no se mueven
cuando aparece un valor gigante, y la desviación estándar sí. Por eso, en una variable sesgada, el IQR
describe mejor a la mayoría, y es además la base de la regla de outliers de la tarea 5.

**Los comandos.**

```python
q1 = df['columna'].quantile(0.25)
q3 = df['columna'].quantile(0.75)
iqr = q3 - q1
```

**Lo que decide usted.** Cómo se escribe un percentil: `.quantile()` recibe una **fracción entre 0 y
1**, no un entero. `0.25`, no `25`. Escribirlo mal no da error, da un número absurdo.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Calcule Q1, Q3 y la desviacion estandar de producci_n_t, e imprimalos.
# 2. Guarde el IQR en iqr_produccion e imprimalo junto con el rango (min a max).

iqr_produccion = None

In [ ]:
comprobar('T3', None if iqr_produccion is None else round(iqr_produccion, 2))

**Tu respuesta:** compare la desviación estándar con el IQR. La desviación estándar es enorme y el IQR
es pequeño. ¿Cuál de las dos describe mejor al cultivo típico, y por qué difieren tanto?

*Tu respuesta:*

---

## Parte 3 · Pasos 4 y 5 sobre `producci_n_t`

**Qué se practica aquí.** Las dos partes del marco donde ninguna función responde por usted: **qué
forma tiene** la distribución y **qué se hace** con los valores extremos.

**El concepto: las cuatro formas.**

| Forma | Cómo se ve | Relación media-mediana |
|-------|-----------|------------------------|
| Normal | Campana simétrica | media = mediana |
| Sesgada a la derecha | Cola larga hacia los valores altos | media > mediana |
| Sesgada a la izquierda | Cola larga hacia los valores bajos | media < mediana |
| Bimodal | Dos jorobas | no aplica |

**El concepto: los dos gráficos.** El **histograma** reparte los datos en cajones (*bins*) y cuenta
cuántos caen en cada uno: muestra la forma. El **boxplot** dibuja la caja de Q1 a Q3, la mediana como
línea del medio, los bigotes hasta 1,5xIQR, y los puntos sueltos de afuera son los outliers.

**Los comandos.**

```python
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

ejes[0].hist(datos, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ejes[0].axvline(valor, color='red', linestyle='--', label='Media')
ejes[0].set_xlabel('...'); ejes[0].set_ylabel('...'); ejes[0].set_title('...')
ejes[0].legend()

ejes[1].boxplot(datos)
ejes[1].set_ylabel('...'); ejes[1].set_title('...')

plt.tight_layout(); plt.show()
```

**Regla del curso, y sale en la rúbrica:** todo gráfico lleva título, eje x, eje y y unidades.

**Advertencia, para que no la confunda con un error.** El histograma de esta variable va a salir como
una sola barra pegada al cero. Ese **es** el resultado correcto: el sesgo es tan fuerte que todo se
apila contra el eje. Es un hallazgo, no un fallo.

In [ ]:
# TU CÓDIGO AQUÍ
# Histograma y boxplot de producci_n_t, lado a lado.
# 1. Histograma con bins=50, con una linea vertical en la media y otra en la mediana.
# 2. Boxplot de la misma variable.
# 3. Titulo y etiquetas de los dos ejes en ambos graficos.

### Tarea 4 · Clasificar la forma

**La pregunta.** ¿Qué forma tiene la distribución de `producci_n_t`?

**El concepto.** Aquí no hay función que responda. Se mira el histograma, se decide, y la decisión se
contrasta con un número que ya calculó en la tarea 2. Si el gráfico y la razón media/mediana no dicen
lo mismo, algo se leyó mal.

**Los comandos.** Ninguno nuevo: se asigna una palabra a una variable.

```python
forma_produccion = 'una de las cuatro palabras'
```

**Lo que decide usted.** Todo. Escriba **una sola palabra**, en minúscula, de estas cuatro:

`'normal'` · `'derecha'` · `'izquierda'` · `'bimodal'`

In [ ]:
# TU CÓDIGO AQUÍ
# Escriba una de las cuatro palabras entre comillas.

forma_produccion = None

In [ ]:
comprobar('T4', None if forma_produccion is None else forma_produccion.strip().lower())

### Tarea 5 · Contar las jirafas

**La pregunta.** ¿Cuántos registros de `producci_n_t` quedan marcados como outlier por la regla
1.5xIQR?

**El concepto.** Un **outlier** es un valor que se sale del comportamiento del resto. La analogía es
el parque de perros con chihuahuas, beagles y labradores donde alguien mete una jirafa: sobresale
porque no pertenece al grupo. La pregunta que importa no es si sobresale, es **por qué**: ¿es un error
de captura o es un caso real inusual?

La regla:

- Límite inferior = Q1 - 1,5 x IQR
- Límite superior = Q3 + 1,5 x IQR

Todo lo que quede afuera es **candidato**, no condenado. Y se usa IQR en vez de desviaciones estándar
porque los cuartiles no se dejan inflar por los propios valores extremos que se quiere detectar.

**Los comandos.**

```python
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
mascara = (df['columna'] < limite_inferior) | (df['columna'] > limite_superior)
n = mascara.sum()          # cuantos son
outliers = df[mascara]     # la tabla con esas filas
```

**Lo que decide usted.** Que se pide el **conteo** de filas marcadas, no el porcentaje ni la tabla. Y
los paréntesis en cada condición del `|`: sin ellos, el código no corre.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Calcule los dos limites de la regla 1.5xIQR e imprimalos.
# 2. Construya la mascara de outliers y guarde el conteo en n_outliers_produccion.
# 3. Imprima el conteo y el porcentaje que representa del total.

n_outliers_produccion = None

In [ ]:
comprobar('T5', n_outliers_produccion)

### Mírele la cara a los outliers

Contar no sirve de nada si no se investiga quiénes son. `.nlargest(n, columna)` devuelve las n filas
con el valor más alto de esa columna, con todas sus demás columnas al lado. Esta celda ya está
escrita: ejecútela y **léala**, porque la tarea siguiente es una decisión y esto es la evidencia.

In [ ]:
print('Los 10 registros de mayor produccion:')
print(df.nlargest(10, 'producci_n_t')[
    ['cultivo', 'departamento', 'municipio', 'a_o', 'rea_sembrada_ha', 'producci_n_t']
].to_string(index=False))

**Tu respuesta:** ¿se eliminan estos outliers? Argumente con lo que acaba de ver: qué cultivo es, en
qué departamento, y con cuánta área sembrada. ¿Es un error de captura o es un caso real inusual? ¿Qué
le pasaría al análisis si los borrara?

*Tu respuesta:*

---

## Parte 4 · Las otras dos variables, con la función ya escrita

**Qué cambia aquí.** Nada de concepto: exactamente los mismos cinco pasos que acaba de hacer a mano,
pero envueltos en una función. Esa es la idea de una función: escribir una vez un procedimiento que
se va a repetir, y llamarlo con distintos datos.

**La función `analisis_univariado(df, variable)` ya está escrita. Usted no la reescribe: la lee y la
llama.** La celda que completa es de dos líneas.

Léala con calma: es el mismo recorrido de las partes 2 y 3, y reconocerlo es parte del ejercicio.
Fíjese en un detalle: usa `.dropna()` antes de graficar, porque matplotlib no dibuja valores
faltantes. Los cálculos de pandas (`.mean()`, `.median()`, `.quantile()`) sí los ignoran solos.

**Qué devuelve.** Un **diccionario**: una colección de pares nombre-valor, donde cada valor se pide
por su nombre entre corchetes. `resultado['razon']` saca la razón media/mediana. Por eso hay que
**guardar** lo que la función devuelve: si solo la llama, imprime todo y se pierde el resultado.

In [ ]:
def analisis_univariado(df, variable):
    """
    Aplica el marco univariado de 5 pasos a una variable numerica.

    Parametros
    ----------
    df : DataFrame
    variable : str, nombre de la columna a analizar

    Devuelve
    --------
    dict con los resultados, para la tabla resumen del final.
    """
    print('=' * 62)
    print(f'ANALISIS UNIVARIADO: {variable}')
    print('=' * 62)

    # --- Paso 1: identificar ---------------------------------------
    print('\n--- PASO 1: IDENTIFICAR ---')
    faltantes = df[variable].isna().sum()
    print(f'Tipo de dato: {df[variable].dtype}')
    print(f'Registros totales: {len(df):,}')
    print(f'Valores no nulos: {df[variable].count():,}')
    print(f'Valores faltantes: {faltantes:,} ({faltantes / len(df) * 100:.2f}%)')

    # --- Paso 2: resumir -------------------------------------------
    print('\n--- PASO 2: RESUMIR (tendencia central) ---')
    media = df[variable].mean()
    mediana = df[variable].median()
    moda = df[variable].mode()[0] if len(df[variable].mode()) > 0 else np.nan
    razon = media / mediana if mediana != 0 else np.nan

    print(f'Media:   {media:,.2f}')
    print(f'Mediana: {mediana:,.2f}')
    print(f'Moda:    {moda:,.2f}')
    print(f'Razon media/mediana: {razon:.2f}')

    # --- Paso 3: dispersar -----------------------------------------
    print('\n--- PASO 3: DISPERSAR ---')
    desviacion = df[variable].std()
    q1 = df[variable].quantile(0.25)
    q3 = df[variable].quantile(0.75)
    iqr = q3 - q1

    print(f'Desviacion estandar: {desviacion:,.2f}')
    print(f'Q1 (percentil 25): {q1:,.2f}')
    print(f'Q3 (percentil 75): {q3:,.2f}')
    print(f'IQR: {iqr:,.2f}')
    print(f'Rango: {df[variable].min():,.2f} a {df[variable].max():,.2f}')

    # --- Paso 4: visualizar ----------------------------------------
    print('\n--- PASO 4: VISUALIZAR ---')
    datos = df[variable].dropna()   # matplotlib no dibuja faltantes

    fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

    ejes[0].hist(datos, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ejes[0].axvline(media, color='red', linestyle='--', linewidth=2,
                    label=f'Media: {media:,.0f}')
    ejes[0].axvline(mediana, color='green', linestyle='-', linewidth=2,
                    label=f'Mediana: {mediana:,.0f}')
    ejes[0].set_xlabel(variable)
    ejes[0].set_ylabel('Frecuencia (numero de registros)')
    ejes[0].set_title(f'Distribucion de {variable}')
    ejes[0].legend()

    ejes[1].boxplot(datos, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.7))
    ejes[1].set_ylabel(variable)
    ejes[1].set_title(f'Diagrama de caja de {variable}')
    ejes[1].set_xticklabels([variable])

    plt.tight_layout()
    plt.show()

    # --- Paso 5: detectar ------------------------------------------
    print('\n--- PASO 5: DETECTAR OUTLIERS (regla 1.5xIQR) ---')
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    outliers = df[(df[variable] < limite_inferior) | (df[variable] > limite_superior)]

    print(f'Limite inferior: {limite_inferior:,.2f}')
    print(f'Limite superior: {limite_superior:,.2f}')
    print(f'Outliers: {len(outliers):,} ({len(outliers) / len(df) * 100:.2f}%)')
    print(f'  por debajo: {(df[variable] < limite_inferior).sum():,}')
    print(f'  por encima: {(df[variable] > limite_superior).sum():,}')

    if razon > 1.2:
        forma, medida = 'Sesgada a la derecha', 'Mediana'
    elif razon < 0.8:
        forma, medida = 'Sesgada a la izquierda', 'Mediana'
    else:
        forma, medida = 'Aproximadamente simetrica', 'Media'

    print('\n--- RESUMEN ---')
    print(f'Forma de la distribucion: {forma}')
    print(f'Medida central recomendada: {medida}')
    print('=' * 62)

    return {'variable': variable, 'media': media, 'mediana': mediana, 'razon': razon,
            'desviacion': desviacion, 'iqr': iqr, 'forma': forma,
            'outliers_pct': len(outliers) / len(df) * 100, 'medida_recomendada': medida}


print('Funcion definida. Ahora usela.')

### Tarea 6 · `rea_sembrada_ha`, el área sembrada

**La pregunta.** ¿Cómo está repartida la tierra cultivada en Colombia? ¿Se parece a la producción?

**El concepto.** La misma pregunta que en las partes 2 y 3, sobre otra variable. Lo único nuevo es que
el trabajo lo hace la función y usted se queda con el resultado.

**Los comandos.**

```python
resultado = analisis_univariado(df, 'nombre_de_la_columna')
resultado['razon']    # la razon media/mediana que trae el diccionario
```

Las otras claves disponibles son `'media'`, `'mediana'`, `'desviacion'`, `'iqr'`, `'forma'`,
`'outliers_pct'` y `'medida_recomendada'`.

**Lo que decide usted.** Cuál de las columnas del dataset es el área sembrada. Recuerde que el nombre
real perdió una letra.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Llame la funcion sobre la columna del area sembrada y guarde el resultado
#    en resultado_area.
# 2. Guarde en razon_area la razon media/mediana que trae ese resultado.

razon_area = None

In [ ]:
comprobar('T6', None if razon_area is None else round(razon_area, 2))

**Tu respuesta:** compare esta distribución con la de la producción. ¿Se parecen? ¿Qué dice eso sobre
cómo está repartida la tierra cultivable en Colombia?

*Tu respuesta:*

### Tarea 7 · `rendimiento_t_ha`, el rendimiento

**La pregunta.** El rendimiento es un **cociente**: producción dividida por área. ¿Se comporta distinto
de las otras dos por serlo?

**El concepto.** Producción y área son magnitudes **absolutas**: crecen con el tamaño del cultivo. El
rendimiento es una magnitud **relativa**: ya está normalizado por tamaño, así que un latifundio y una
huerta pueden tener el mismo rendimiento. Es la misma idea del demo, cuando el consumo total por
estrato y el consumo por suscriptor daban órdenes opuestos.

**Los comandos.** Los mismos de la tarea 6.

**Lo que decide usted.** Nada de técnica. Lo que se evalúa aquí es la comparación: mire la razón de
esta variable al lado de las otras dos antes de escribir la interpretación.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Llame la funcion sobre la columna de rendimiento y guarde el resultado
#    en resultado_rendimiento.
# 2. Guarde en razon_rendimiento la razon media/mediana de ese resultado.

razon_rendimiento = None

In [ ]:
comprobar('T7', None if razon_rendimiento is None else round(razon_rendimiento, 2))

Antes de interpretar, mire quiénes son los outliers de esta variable. La celda ya está escrita.

In [ ]:
print('Los 6 rendimientos mas altos:')
print(df.nlargest(6, 'rendimiento_t_ha')[
    ['cultivo', 'departamento', 'municipio', 'rea_sembrada_ha', 'producci_n_t', 'rendimiento_t_ha']
].to_string(index=False))

**Tu respuesta, en tres partes:**

1. Compare las tres razones media/mediana (producción, área, rendimiento). ¿Por qué la del rendimiento
   es tanto menor? ¿Qué tiene de distinto un cociente?

*Tu respuesta:*

2. En las dos variables anteriores la respuesta sobre los outliers fue "se conservan, son cultivos
   industriales reales". Mire la tabla de arriba: el mismo cultivo, en el mismo departamento, con
   áreas de 3 a 6 hectáreas, repitiendo valores altísimos. ¿Sigue siendo la misma respuesta?

*Tu respuesta:*

3. Un cociente se dispara por dos motivos distintos: porque el numerador es muy grande, o porque el
   denominador es muy chico. ¿Cómo verificaría cuál de los dos está pasando aquí, y a quién le
   preguntaría?

*Tu respuesta:*

---

## Parte 5 · GroupBy: comparar entre grupos

**Qué se practica aquí.** Pasar de "el promedio es X" a "el promedio del grupo A es X y el del grupo B
es Y". Lo segundo es infinitamente más útil: un número único describe a todo el mundo y no describe a
nadie.

**El concepto: los M&Ms.** Se separan en montoncitos por color (**split**), se cuenta cada montoncito
(**apply**), y se ponen los conteos lado a lado (**combine**).

**Un GroupBy son siempre tres decisiones:**

1. ¿Por cuál columna separo? — la categórica.
2. ¿Cuál columna resumo? — la numérica.
3. ¿Con cuál operación? — `.mean()`, `.sum()`, `.count()`, `.median()`.

**Los comandos.**

```python
serie = df.groupby('COLUMNA_GRUPO')['COLUMNA_VALOR'].mean()
serie.sort_values(ascending=False)   # ordenar de mayor a menor
serie.idxmax()   # la etiqueta donde esta el maximo (aqui, el nombre del grupo)
serie.max()      # el valor maximo
```

**La costumbre que no se negocia:** ordene siempre. GroupBy ordena alfabéticamente por defecto, y el
hallazgo casi nunca está en orden alfabético.

### Tarea 8 · Producción media por grupo de cultivo

**La pregunta.** ¿Cuál grupo de cultivo produce más toneladas en promedio, y cuántas?

**El concepto.** Las tres decisiones de los M&Ms aplicadas por primera vez sobre este dataset. Se
resume con la media porque la pregunta es por el promedio del grupo, y después se ordena.

**Los comandos.** Los de la parte 5, todos.

**Lo que decide usted.** Las tres decisiones. Guarde en `grupo_lider` una lista de dos elementos:
`[nombre_del_grupo, produccion_media]`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. GroupBy por grupo_de_cultivo sobre la produccion, con la media.
# 2. Ordene de mayor a menor e imprima la tabla completa (son 13 grupos).
# 3. Guarde en grupo_lider la lista [nombre, valor] del primero.

grupo_lider = None

In [ ]:
comprobar('T8', None if grupo_lider is None
          else [grupo_lider[0], round(float(grupo_lider[1]), 2)])

**Tu respuesta:** mire la tabla completa, no solo el primero. FLORES Y FOLLAJES queda a mitad de
tabla, pese a ser uno de los renglones de exportación más importantes del país. ¿Por qué? Pista: piense
en qué unidad mide este dataset y qué unidad le importaría a un exportador.

*Tu respuesta:*

### Tarea 9 · Rendimiento mediano por ciclo de cultivo

**La pregunta.** ¿Cuál ciclo de cultivo rinde más por hectárea, y cuánto?

**El concepto.** Aquí se pide la **mediana**, no la media, y no es un capricho: en la tarea 7 usted
comprobó que el rendimiento está sesgado, y la regla del curso dice que en variables sesgadas se
reporta la mediana. La operación del GroupBy se elige con el mismo criterio con que se elige la medida
de centro de una variable suelta.

**Los comandos.** Los mismos de la tarea 8, cambiando la operación.

**Lo que decide usted.** La operación, y contra qué columna. Guarde en `ciclo_lider` la lista
`[nombre_del_ciclo, rendimiento_mediano]`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. GroupBy por ciclo_de_cultivo sobre el rendimiento, con la mediana.
# 2. Ordene de mayor a menor e imprima los tres ciclos.
# 3. Guarde en ciclo_lider la lista [nombre, valor] del primero.

ciclo_lider = None

In [ ]:
comprobar('T9', None if ciclo_lider is None
          else [ciclo_lider[0], round(float(ciclo_lider[1]), 2)])

**Tu respuesta:** las tres medianas están bastante cerca entre sí. ¿Diría que el ciclo de cultivo
explica el rendimiento? ¿Qué otra variable de este dataset le parece más prometedora para explicarlo, y
por qué?

*Tu respuesta:*

---

## Parte 6 · Una pregunta que usted arma sola

**Qué cambia aquí.** Nada de técnica: esta tarea no usa un solo comando que no haya usado ya. Lo que
cambia es que **la pregunta viene en español y el ensamblaje es suyo**. Le damos la lista de comandos;
no le damos el orden.

Es deliberado: en los momentos evaluativos nadie le va a dar la secuencia.

Los comandos disponibles son estos, todos vistos:

```python
df.groupby('COLUMNA')['VALOR'].sum()        df.groupby('COLUMNA')['VALOR'].median()
serie.nlargest(n)                           serie.index
df['columna'].isin([...])                   df[mascara]
serie.sort_values(ascending=False)          serie.idxmax()   serie.max()
```

Si se atasca, parta el problema: resuelva **una** pieza, imprímala, mírela, y solo entonces añada la
siguiente.

### Tarea 10 · Los grandes productores, ¿son los más eficientes?

**La pregunta.** Tome los **5 departamentos con mayor producción total** del país. Entre esos cinco,
¿cuál tiene el **rendimiento mediano** más alto, y cuál es ese rendimiento?

**El concepto.** Producir mucho y producir bien no son lo mismo. Un departamento puede encabezar la
producción total simplemente por tener más tierra sembrada, con un rendimiento por hectárea mediocre.
Separar volumen de eficiencia es exactamente lo que hace un analista, y es la misma lección del giro
del demo: una suma grande puede ser solo un grupo grande.

**Lo que decide usted.** El orden de las tres piezas: qué se agrupa primero, qué se filtra con qué, y
sobre cuál tabla se calcula la mediana. Guarde en `depto_lider` la lista
`[nombre_del_departamento, rendimiento_mediano]`.

Una advertencia: el rendimiento mediano hay que calcularlo **sobre las filas del DataFrame original**
de esos cinco departamentos. No se puede sacar de la tabla de sumas.

In [ ]:
# TU CÓDIGO AQUÍ
# Pieza 1: produccion total por departamento, y quedese con los 5 mas altos.
# Pieza 2: filtre el DataFrame original para dejar solo esos cinco departamentos.
# Pieza 3: sobre esas filas, rendimiento mediano por departamento, ordenado.
# Guarde en depto_lider la lista [nombre, valor] del primero e imprima las dos tablas.

depto_lider = None

In [ ]:
comprobar('T10', None if depto_lider is None
          else [depto_lider[0], round(float(depto_lider[1]), 2)])

**Tu respuesta:** el departamento que encabeza la producción total, ¿es el mismo que encabeza el
rendimiento? Escriba dos frases de conclusión, y cuide la trampa: estos datos son de producción
reportada por municipio, no de productividad medida en campo. ¿Qué **no** se puede concluir con ellos?

*Tu respuesta:*

---

## Punto de control

Ejecute la celda de abajo para ver cuántas de las diez tareas quedaron correctas.

Si alguna sigue pendiente, no pase de largo: la clase 5 arranca dando por sabido todo esto. Si está en
el salón, levante la mano ahora, que el profesor está aquí para eso.

In [ ]:
resumen_puntos_de_control()

---

## Tabla resumen

Llene esta tabla con los números que ya calculó. Es el entregable en una sola vista, y es lo primero
que se mira al revisar.

| Variable | Media | Mediana | Razón media/mediana | Forma | Outliers (%) | Medida recomendada | Decisión sobre los outliers |
|----------|-------|---------|---------------------|-------|--------------|--------------------|-----------------------------|
| `producci_n_t` | | | | | | | |
| `rea_sembrada_ha` | | | | | | | |
| `rendimiento_t_ha` | | | | | | | |

**Y debajo, una frase por variable, sin ningún número.** Tres frases en total. Una frase que se pueda
decir en voz alta en una reunión donde nadie ha visto el notebook.

*Tu respuesta:*

---

## Reflexión (en casa)

Responda en español, dos o tres frases por pregunta.

**1. ¿Cuál de las diez tareas le costó más y por qué?**

*Tu respuesta:*

**2. Las tres variables salieron sesgadas a la derecha, pero no en el mismo grado.** Si mañana le
entregan una variable nueva de un dataset que no conoce, ¿en qué orden haría los cinco pasos y en cuál
de ellos se detendría más? Justifique.

*Tu respuesta:*

**3. Piense en el dataset que su equipo eligió para el proyecto.** Escriba **una** variable numérica
suya, qué forma espera que tenga su distribución y por qué, y qué haría si aparecen outliers (no tiene
que ejecutar nada, solo escribirlo).

*Tu respuesta:*

---

## Opcional · Solo si terminó todo

No se comprueban ni entran en la retroalimentación.

**A. El histograma escondido.** El de `producci_n_t` sale como una sola barra porque el sesgo es
brutal. En escala logarítmica aparece la forma que el sesgo esconde. Filtre los valores mayores que
cero, aplique `np.log10()` y grafique eso.

**B. Los tres boxplots lado a lado**, para comparar las tres variables de un vistazo.
`plt.subplots(1, 3, figsize=(15, 5))` y un bucle.

**C. Producción total por departamento, los 10 primeros**, en un gráfico de barras horizontales
(`.plot(kind='barh')`).

In [ ]:
# OPCIONAL — TU CÓDIGO AQUÍ

---

## Antes de entregar

1. **Kernel → Restart and Run All.** Si algo revienta, arréglelo. Un cuaderno que no corre de arriba a
   abajo le pone techo a la dimensión Hacer.
2. Verifique que **todas** las celdas `Tu respuesta:` están escritas. El número no es el análisis.
3. Revise que los gráficos que usted dibujó tienen título y los dos ejes etiquetados.
4. Guarde como `clase04_reto_APELLIDO.ipynb` y súbalo al aula virtual, antes del inicio de la clase 5.